# Quick Phase 7 multiwindow refinement (no training, no checkpoint upload)

Upload the `submission.zip` from your 0.4454 multiwindow run when prompted. It is under `MyDrive/CS781_PHASE7_OUTPUT/P7_E23_multiwindow_manual/`; download that small ZIP to your computer first. The notebook downloads the official Test cases with your HF token, makes conservative sparse-retrieval swaps, and downloads a new `submission.zip`. It does not need the 641 MiB `.pt` file or a GPU. The change is unvalidated on hidden Test labels and can lower the score; keep your original 0.4454 ZIP.


In [ ]:
%pip -q install "huggingface_hub>=0.36,<3" scikit-learn

In [ ]:
from google.colab import files
import io, json, zipfile

print("Upload the MULTIWINDOW submission.zip that scored 0.4454 (a small ZIP, not the 641 MiB .pt file).")
uploaded = files.upload()
assert len(uploaded) == 1, "Upload one file: multiwindow submission.zip or prediction.json"
name, payload = next(iter(uploaded.items()))
if name.lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        assert "prediction.json" in archive.namelist(), "ZIP needs prediction.json at its root"
        original = json.loads(archive.read("prediction.json"))
elif name.lower().endswith(".json"):
    original = json.loads(payload)
else:
    raise ValueError("Select the multiwindow submission.zip or prediction.json")
del payload, uploaded
assert isinstance(original, dict) and len(original) == 237
assert all(isinstance(qid, str) and isinstance(ids, list) and len(ids) == 7
           and len(set(ids)) == 7 and all(isinstance(cid, str) for cid in ids)
           for qid, ids in original.items())
print("Loaded the 237-query multiwindow submission from:", name)


In [ ]:
from getpass import getpass
from huggingface_hub import hf_hub_download
from google.colab import userdata
from pathlib import Path
import numpy as np
import re, gc
from sklearn.feature_extraction.text import TfidfVectorizer

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
REVISION = "93b0059be756c6c461f3a225de25aabda7d5f84a"
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Approved Hugging Face read token (hidden): ").strip()
assert HF_TOKEN, "The official Phase 7 dataset needs your approved HF token"

def official(name):
    path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset",
                           filename="Phase_7_PCR/Test/" + name,
                           revision=REVISION, token=HF_TOKEN)
    return json.loads(Path(path).read_text(encoding="utf-8"))

queries = official("test.json")
candidates = official("test_candidates.json")
assert len(queries) == 237 and len(candidates) == 1727
assert set(original) == set(queries)
assert all(qid in candidates and all(cid in candidates and cid != qid for cid in ids)
           for qid, ids in original.items())
qids, cids = list(queries), list(candidates)
cindex = {cid: i for i, cid in enumerate(cids)}
print("Official Test data loaded and every uploaded prediction ID validated")


In [ ]:
# Recompute the measured sparse retrieval signals on the official Test pool.
docs = [" ".join(case["text"])[:100_000] for case in candidates.values()]
qdocs = [" ".join(case["text"])[:100_000] for case in queries.values()]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=240_000,
                             stop_words="english", sublinear_tf=True, dtype=np.float32)
matrix = vectorizer.fit_transform(docs + qdocs)
full = (matrix[len(cids):] @ matrix[:len(cids)].T).toarray().astype(np.float32)
del matrix, vectorizer
gc.collect()
print("Full-document scores ready")

chunks, starts = [], []
for document in docs:
    starts.append(len(chunks))
    chunks.extend(document[j:j+3000] for j in range(0, len(document), 3000))
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=240_000,
                             stop_words="english", sublinear_tf=True, dtype=np.float32)
matrix = vectorizer.fit_transform(chunks + qdocs)
raw = (matrix[len(chunks):] @ matrix[:len(chunks)].T).toarray().astype(np.float32)
passage = np.maximum.reduceat(raw, starts, axis=1)
del matrix, vectorizer, raw, chunks
gc.collect()
print("Passage scores ready")

CITE_A = re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})\s*(AIR|SCR|SCC|SCALE)\s*(?:\(\d+\))?\s*(\d{1,5})\b", re.I)
CITE_B = re.compile(r"\bAIR\s*(18\d{2}|19\d{2}|20\d{2})\s*SC\s*(\d{1,5})\b", re.I)
def citations(text):
    keys = {(m.group(1), m.group(2).upper(), int(m.group(3))) for m in CITE_A.finditer(text)}
    keys.update((m.group(1), "AIR", int(m.group(2))) for m in CITE_B.finditer(text))
    return keys
qref = [citations(document) for document in qdocs]
cref = [citations(document[:1800]) for document in docs]
cite = np.asarray([[bool(a & b) for b in cref] for a in qref], dtype=np.float32)
sparse = full + 0.5 * passage + 0.1 * cite
for i, qid in enumerate(qids):
    sparse[i, cindex[qid]] = -1e3
print("Sparse scores ready:", sparse.shape)


In [ ]:
# Change at most one low-ranked multiwindow choice per query.
# This is a heuristic on unlabeled Test data; retain the 0.4454 ZIP as a fallback.
def refine(max_outside_rank, min_last_rank, min_gap, max_swaps):
    result = {qid: list(original[qid]) for qid in qids}
    proposals = []
    for i, qid in enumerate(qids):
        current = result[qid]
        order = np.argsort(-sparse[i])
        sparse_rank = np.empty(len(cids), dtype=np.int32)
        sparse_rank[order] = np.arange(len(cids))
        last = current[-1]
        outsider = next((cids[j] for j in order[:max_outside_rank]
                         if cids[j] not in current and cids[j] != qid), None)
        if outsider is not None:
            last_j, outside_j = cindex[last], cindex[outsider]
            gap = float(sparse[i, outside_j] - sparse[i, last_j])
            if sparse_rank[last_j] >= min_last_rank and gap >= min_gap:
                proposals.append((gap, qid, last, outsider))
    swaps = []
    for gap, qid, last, outsider in sorted(proposals, reverse=True)[:max_swaps]:
        result[qid][-1] = outsider
        swaps.append((qid, last, outsider, round(gap, 4)))
    assert all(len(ids) == len(set(ids)) == 7 for ids in result.values())
    return result, swaps

recommended, strict_swaps = refine(max_outside_rank=3, min_last_rank=80,
                                   min_gap=0.15, max_swaps=25)
moderate, moderate_swaps = refine(max_outside_rank=5, min_last_rank=25,
                                  min_gap=0.10, max_swaps=70)
print("Conservative changes:", len(strict_swaps), "of", len(qids), "queries")
print("Moderate changes:", len(moderate_swaps), "of", len(qids), "queries")
print("First conservative changes:", strict_swaps[:5])

def write_zip(predictions, filename):
    raw = json.dumps(predictions, ensure_ascii=False, indent=1).encode("utf-8")
    target = Path("/content") / filename
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("prediction.json", raw)
    with zipfile.ZipFile(target) as archive:
        assert archive.namelist() == ["prediction.json"] and archive.testzip() is None
        check = json.loads(archive.read("prediction.json"))
    assert set(check) == set(queries)
    assert all(len(ids) == len(set(ids)) == 7 and
               all(cid in candidates and cid != qid for cid in ids)
               for qid, ids in check.items())
    return target

main_zip = write_zip(recommended, "submission.zip")
moderate_zip = write_zip(moderate, "submission_moderate.zip")
print("Ready:", main_zip, "and", moderate_zip)
files.download(str(main_zip))


In [ ]:
# Optional: run only if you want to test the less conservative version too.
files.download("/content/submission_moderate.zip")
